In [ ]:
import requests
import pandas as pd
import time


In [ ]:
def OpenAlexAPI(url, params):
    
    # Initialize cursor
    cursor = "*"

    # Loop through pages
    all_datasets = []
    page = 0
    while cursor:
        params["cursor"] = cursor
        
        # Check for request success
        response = requests.get(url, params=params)
        if response.status_code == 200:
            print(f"Status code: {response.status_code}", ';',  "Page number: ", page)
        if response.status_code != 200:
            print("Oh no! Something went wrong during the live demo! How embarrassing!")
            response.raise_for_status()
        records = response.json()['results']
        record_number = 0
        page += 1
        for record in records:
            record_number += 1
            #print("Page number: ", page, 'Record number:', record_number, record)
            all_datasets.append({'id': record['id'],
                                'doi': record['doi'],
                                'title': record['title'],
                                'publication_year': record['publication_year'],
                                'publiation_date': record['publication_date'],
                                'URL': record['primary_location']['landing_page_url'],
                                'language': record['language'],
                                'Primary Location': record['primary_location']['source'],
                                #'Host Source': record['primary_location']['source'],
                                'is_oa': record['open_access']['is_oa'],
                                'oa_status': record['open_access']['oa_status'],
                                'type': record['type'],
                                'index location': record['indexed_in'],
                                'Authors': record['authorships'][:],
                                'author ID': record['authorships'][0]['author']['id'],
                                
                                
                                
                                
                                #'author_position': record['authorships'][0]['author_position'],
                                # 'author': record['authorships'][0]['author']['display_name'],
                                # 'Orcid ID': record['authorships'][0]['author']['orcid'],
                                #'Institution id': record['primary_location']['source']['id'],
                               # 'Institution': record['authorships'][0]['institutions'],
                                #'ror': record['authorships'][0]['Institutions']['ror'],
                               # 'linneage': record['authorships']['Institutions']['lineage'],

                                #'type test': record['type'][:],
                                #'Instiution URL Code': url,
                                
                                #'Index Location': record['type']['indexed_in'][0]
                                #"index location": record['type'][0],

                                })

                                #'Institution': record['institutions'][0],
                                
                            
                                #'Institution': record['institutions'][0],

                                #'authorships 3': record['authorships'][1]['institutions']['display_name'],
                                #'authorships 3': record.get("authorships")[0].get('institutions')[1],
                                #'institution 4': record['authorships'][0]['institutions'],
                                #'institution 5': record['authorships'][2][2]
                                
    
            

        # Update cursor
        cursor = response.json()['meta']['next_cursor']
        
        time.sleep(3)
    return all_datasets

    
# types/dataset,
parameters = {
    #"filter": f"authorships.institutions.lineage:i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549",  # University of Tasmania
    "per-page": 100,
    #"select": "id,doi,publication_year,title,primary_location,authorships,topics",
}


In [ ]:
MN_df = OpenAlexAPI('https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i130238516', parameters)

url_list = 'https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i130238516', 'https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i188538660'
for i in url_list:
    print(i)
    MN_df = OpenAlexAPI(i, parameters)

In [ ]:
def CleanJSON(column,InputDF):
    df = pd.concat([pd.DataFrame(x) for x in InputDF[column]], keys=InputDF.index).reset_index(level=1,drop=True)
    return df

In [ ]:
pd.set_option("display.max_columns", 55)


In [ ]:
#new_test = pd.DataFrame(mn_test)
MN_df =pd.json_normalize(MN_df)
MN_df

In [ ]:
MN_df.columns

In [ ]:
MN_df = MN_df[['id', 'doi', 'title', 'publication_year', 'publiation_date', 'URL',
       'language', 'is_oa', 'oa_status', 'author ID', 'type', 'index location',
       'Authors', 'Primary Location.id', 'Primary Location.display_name',
       'Primary Location.issn_l', 'Primary Location.is_oa', 'Primary Location.is_in_doaj',
       'Primary Location.type']]

In [ ]:
MN_df

In [ ]:
MN_df.to_csv('test.csv')

In [ ]:
MN_df = pd.read_csv('test.csv').drop('Unnamed: 0', axis=1)
MN_df

In [ ]:
MN_df = MN_df['index location'].str[0]
MN_df

In [ ]:
#MN_df = pd.concat([MN_df, index_df], axis=1)
#MN_df

In [ ]:
authors_df = CleanJSON('Authors', MN_df)
authors_df

In [ ]:
doi_df = MN_df[['id', 'doi', 'title', 'publication_year', 'publiation_date', 'URL',
       'language', 'is_oa', 'oa_status', 'type', 'index location', 'Primary Location.id', 'Primary Location.display_name',
       'Primary Location.issn_l', 'Primary Location.is_oa',
       'Primary Location.is_in_doaj', 'Primary Location.type']]
doi_df

In [ ]:
MN_df.columns

In [ ]:
authors_df = pd.concat([authors_df, doi_df], axis=1)
authors_df


In [ ]:
authors_df = authors_df.rename(columns={'raw_author_name': 'author_name'})

In [ ]:
authors_df = authors_df.reset_index()


In [ ]:
author_info_df= pd.json_normalize(authors_df['author'])
author_info_df = author_info_df[['id', 'orcid']]
author_info_df = author_info_df.rename(columns={'id': 'author_id'})
author_info_df

In [ ]:
authors_df = pd.concat([authors_df, author_info_df], axis=1).reset_index(drop=True)
authors_df.head()

In [ ]:
authors_df

In [ ]:
#def CleanJSON2(column,InputDF):
#   df = pd.concat([pd.DataFrame(x) for x in InputDF[column]], keys=InputDF.index).reset_index()
#    return df
# create multi index?
#authors_df = authors_df.set_index(['index', 'doi'])



#df2 = pd.concat([pd.DataFrame(x) for x in authors_df['institutions']], keys=authors_df.index)
#df2


In [ ]:
institution_df = CleanJSON('institutions', authors_df)

# remove list format for lineage
institution_df['lineage'] = institution_df['lineage'].str[0]
institution_df

In [ ]:
institution_df = institution_df.rename(columns={'id':'institution_id', 'display_name': 'institution_name', 'type': 'instiution_type'})
authors_df = authors_df.rename(columns={'display_name': 'author_name','id': 'author_id'})



In [ ]:
institution_df

In [ ]:
institution_df.groupby('lineage').size().sort_values(ascending=False).head(25)



In [ ]:
affilliations_df = CleanJSON('affiliations', authors_df)
affilliations_df['institution_ids'] = affilliations_df['institution_ids'].str[0]
affilliations_df = affilliations_df.rename(columns={'institution_ids': 'lineage'})
affilliations_df

In [ ]:
institution_df = institution_df.reset_index()
affilliations_df = affilliations_df.reset_index()

In [ ]:
df = [institution_df, affilliations_df ]
institutions_df = pd.concat(df, axis=1)
institutions_df


In [ ]:
authors_df

In [ ]:
#authors_df = authors_df[['index', 'author_name', 'author_id', 'orcid', 'author_position', 'is_corresponding', 'institutions', 'countries', 'doi']]
authors_df = authors_df[['author_name', 'author_id', 'orcid', 'author_position', 'is_corresponding', 'countries', 'doi', 'title', 'publication_year', 'publiation_date', 'URL',
       'language', 'is_oa', 'oa_status', 'type', 'index location', 'Primary Location.id', 'Primary Location.display_name',
       'Primary Location.issn_l', 'Primary Location.is_oa', 'Primary Location.is_in_doaj',
       'Primary Location.type']]
authors_df = authors_df.loc[:,~authors_df.columns.duplicated()].copy()
authors_df


In [ ]:
authors_df = authors_df.reset_index()
institutions_df = institutions_df.reset_index()


In [ ]:
author_inst_df = [authors_df, institutions_df ]
final_author_inst_df = pd.concat(author_inst_df, axis=1)
final_author_inst_df.columns

In [ ]:
final_author_inst_df = final_author_inst_df[['author_name', 'author_id', 'orcid', 'author_position',
       'is_corresponding', 'countries', 'doi', 'title', 'publication_year',
       'publiation_date', 'URL', 'language', 'is_oa', 'oa_status', 'type',
       'index location', 'Primary Location.id',
       'Primary Location.display_name', 'Primary Location.issn_l',
       'Primary Location.is_oa', 'Primary Location.is_in_doaj',
       'Primary Location.type',  'institution_id',
       'institution_name', 'ror', 'country_code', 'instiution_type', 'lineage', 'raw_affiliation_string', 'lineage']]

In [ ]:
# set author ror id to institutions
final_author_inst_df = final_author_inst_df[final_author_inst_df['ror'] == 'https://ror.org/017zqws13']
final_author_inst_df

In [ ]:
final_author_inst_df.to_csv('OpenAlexMNData.csv')

In [ ]:
# set ror ids for all institutions
final_author_inst_df = final_author_inst_df[(final_author_inst_df['ror'] == 'https://ror.org/017zqws13') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295')
                                            | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295')
                                            | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295')
                                            | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295')
                                            | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295') | (final_author_inst_df['ror'] == 'https://ror.org/i205783295')]
final_author_inst_df

In [ ]:
i205783295|i2801744472|i170897317|i145311948|i2802508227|i57206974|i130769515|i20089843|i2801919071|i188538660|i79576946|i27837315|i2800403580|i4210130704|i114395901|i169521973|i135310074|i859038795|i204465549

In [ ]:
# I added these columns to author_df instead
# MN_df =pd.merge(MN_df, final_author_inst_df, how='left', left_on='doi', right_on='doi')
MN_df

In [ ]:
MN_df.columns

In [ ]:
t = pd.concat([institution_df, affilliations_df], axis=1).reset_index()
t

In [ ]:
authors_df = authors_df[['index', 'author_name', 'author_id', 'orcid', 'author_position', 'is_corresponding', 'institutions', 'countries','raw_affiliation_strings',
       'affiliations','doi']]
authors_df

In [ ]:
affilliations_df = CleanJSON('affiliations', authors_df)
affilliations_df

In [ ]:
t = pd.concat([authors_df, institution_df], axis=1)
t


In [ ]:
affilliations_df['raw_affiliation_string'].unique()

In [ ]:
df = pd.concat([pd.DataFrame(x) for x in authors_df['institutions']], keys=authors_df.index).reset_index(level=1,drop=True)
df


In [ ]:
authors_df.merge(institution_df, how='left')

In [ ]:
authors_df = pd.concat([authors_df, institution_df], axis=1)
authors_df


In [ ]:
test = df2[df2['ror'] == 'https://ror.org/017zqws13']
#test.index = test.index.get_level_values('display_name')
test


In [ ]:
test['display_name'][0][1]

In [ ]:
test.loc[~test.index.get_level_values('display_name').duplicated(keep='last')]




In [ ]:
test.loc[test.index[1]]

In [ ]:
test.loc[test.index.levels[0][1]]


In [ ]:
s = test.groupby(level=[0,1])['display_name'].nth(0)
s



In [ ]:
test = test.set_index(['index'])
test

In [ ]:
test = test.reset_index(drop=True)

In [ ]:
test

In [ ]:
authors_df.merge(df2, left_index=True, right_index=True)



In [ ]:
result = institutions_df.merge(affiliations_df, how='inner', on='index')

result


In [ ]:
authors_df = pd.concat([authors_df, raw_affiliation_strings_df], axis=1)
authors_df 


In [ ]:
authors_df= authors_df[['author_position', 'institutions', 'countries',
       'is_corresponding', 'raw_affiliation_strings',
       'affiliations', 'id', 'display_name', 'orcid']]
authors_df

In [ ]:
affilliations_df = CleanJSON('affiliations', authors_df)
affilliations_df

In [ ]:
index_df = CleanJSON('index location', MN_df)

In [ ]:
authors_df = authors_df.set_index([''])

In [ ]:
authors_df.set_index('author_position').stack().reset_index()

In [ ]:
authors_df = authors_df.set_index('').stack().reset_index()

In [ ]:
authors_df

In [ ]:
institutions_df = CleanJSON('institutions', authors_df)
institutions_df.head(25)

In [ ]:
lineage_df = CleanJSON('lineage', institutions_df)
lineage_df

In [ ]:
institutions_df = institutions_df[['id', 'display_name', 'ror', 'country_code', 'type']]

In [ ]:
institutions_df = pd.merge(institutions_df, lineage_df,  how='inner', left_index=True, right_index=True)
institutions_df.head(25)

In [ ]:
#authors_df = CleanJSON('Authors', MN_df).reset_indeX(drop=True)
#authors_df

In [ ]:
authors_df.reset_index(drop=True).head(25)


In [ ]:
Institutions_df = CleanJSON('institutions', authors_df)
Institutions_df = Institutions_df.rename(columns={'id': 'author_institution_ids', 'display_name': 'author_institution', 'ror': 'author_institution ror', 'lineage':'author institution lineage', 'type': 'institution type'})
Institutions_df = Institutions_df[['author_institution_ids', 'author_institution', 'author_institution ror', 'institution type', 'author institution lineage']]
#Institutions_df = Institutions_df.drop(['country_code'])
raw_affiliation_strings_df = CleanJSON('raw_affiliation_strings', authors_df)
raw_affiliation_strings_df = raw_affiliation_strings_df.rename(columns={0: 'Record Institution Name'})

affiliations_df = CleanJSON('affiliations', authors_df)
affiliations_df = affiliations_df.rename(columns={'institution_ids': 'affiliation_institution_ids'})

Institutions_df


test = authors_df[(authors_df['author_position'] =='middle') & (authors_df['raw_author_name'] == 'Claire Porter')]
test

In [ ]:
Institutions_df = CleanJSON('institutions', authors_df)
Institutions_df = Institutions_df.rename(columns={'id': 'author_institution_ids', 'display_name': 'author_institution', 'ror': 'author_institution ror', 'lineage':'author institution lineage', 'type': 'institution type'})
Institutions_df = Institutions_df[['author_institution_ids', 'author_institution', 'author_institution ror', 'institution type', 'author institution lineage']]
#Institutions_df = Institutions_df.drop(['country_code'])
raw_affiliation_strings_df = CleanJSON('raw_affiliation_strings', authors_df)
raw_affiliation_strings_df = raw_affiliation_strings_df.rename(columns={0: 'Record Institution Name'})

affiliations_df = CleanJSON('affiliations', authors_df)
affiliations_df = affiliations_df.rename(columns={'institution_ids': 'affiliation_institution_ids'})

Institutions_df


In [ ]:
Institutions_df = Institutions_df[['author_institution_ids', 'author_institution',
       'author_institution ror', 'institution type']]
#Institutions_df = Institutions_df.drop_duplicates()
#raw_affiliation_strings_df = raw_affiliation_strings_df.drop_duplicates()
Institutions_df

In [ ]:
Institutions_df =Institutions_df.reset_index()
raw_affiliation_strings_df = raw_affiliation_strings_df.reset_index()
affiliations_df = affiliations_df.reset_index()

In [ ]:
authors_df = authors_df[['author_position', 'raw_author_name']] #.reset_index()
#authors_df = authors_df.rename(columns={"index": "1"})
authors_df

In [ ]:
authors_df.columns

In [ ]:
#df_merge = Institutions_df.join(authors_df, how='outer')
#df_merge.head(45)
pd.concat([authors_df, Institutions_df], axis=1)


In [ ]:
#df_merge= Institutions_df.join(authors_df, how='outer')
#df_merge.head(35)
#result = pd.concat([authors_df, Institutions_df], axis=1)
#result
# rewrite authors keep instiuttion  data
df_merge =authors_df.merge(Institutions_df, on='index',how ='inner')
df_merge

In [ ]:
df_merge2= df_merge.merge(raw_affiliation_strings_df, on='index',how ='inner')
df_merge
df_merge2.head(10)


In [ ]:
df_merge3= df_merge2.merge(affiliations_df, on='index',how ='inner')
df_merge3
df_merge3.head(25)



In [ ]:
df_merge3

In [ ]:
MN_df= MN_df.reset_index()

In [ ]:
df_merge4= df_merge3.merge(MN_df, on='index',how ='inner')
df_merge4
df_merge4.head(25)


In [ ]:
cleaned_MN_df = df_merge4[['id', 'doi','title', 'author', 'Orcid ID', 'author_position', 'publication_year','URL', 'language', 'is_oa', 'oa_status', 'author', 'Orcid ID', 
                               'Primary Location.display_name', 'Primary Location.is_oa', 
                               'Host Source.display_name',  'Host Source.type', 'author_institution_ids',# 'author_institution',
                                 'author_institution ror', 'institution type', 'Record Institution Name']]
cleaned_MN_df

In [ ]:
cleaned_MN_df = cleaned_MN_df.drop_duplicates()
cleaned_MN_df.head(25)

In [ ]:
cleaned_MN_df = cleaned_MN_df.drop_duplicates()
cleaned_MN_df[0:4]

In [ ]:
test = cleaned_MN_df[(cleaned_MN_df['author_position'] =='first') ]
test


In [ ]:
#cleaned_MN_df['author_institution_ids'].isna().sum()ArcticDEM, Version 3

In [ ]:
# filter NaN's on  
cleaned_MN_df = cleaned_MN_df[cleaned_MN_df['title'] == 'The Reference Elevation Model of Antarctica - Mosaics, Version 2']
cleaned_MN_df

In [ ]:
len(cleaned_MN_df)

In [ ]:
frames = (cleaned_MN_df, Institutions_df, raw_affiliation_strings_df)


#MN_df = pd.concat([MN_df,  Institutions_df], axis=1)
#MN_df = pd.concat([MN_df,  affiliations_df], axis=1).reset_index(level=1,drop=True)
#MN_df = pd.concat([MN_df,  affiliations_df], axis=1)

new_frames = pd.concat(frames)
new_frames

In [ ]:
new_frames[0].unique()

In [ ]:
test = new_frames[new_frames['author'] =='I. M. Howat']
test

In [ ]:
combine_MN_df['author_institution'].unique()

In [ ]:
#joined_df = pd.concat([Institutions_df, raw_affiliation_strings_df.set_axis(Institutions_df.index)], axis=1)
#joined_df

In [ ]:
cleaned_MN_df

In [ ]:
cleaned_MN_df = cleaned_MN_df[['id', 'doi','title','publication_year','URL', 'language', 'is_oa', 'oa_status', 'author', 'Orcid ID', 
                               'Primary Location.display_name', 'Primary Location.is_oa', 
                               'Host Source.display_name',  'Host Source.type', 'author_institution_ids', 'author_institution',
                                 'author_institution ror', 'institution type', 'raw_affiliation_string', 0]]

In [ ]:
cleaned_MN_df

In [ ]:
MN_df = MN_df.append(raw_affiliation_strings_df)
MN_df

In [ ]:
MN_df = pd.concat([pd.DataFrame(x) for x in MN_df2['affiliations']], keys=MN_df2.index).reset_index(level=1,drop=True)
MN_df

In [ ]:
def CleanJSON(column,InputDF):
    df = pd.concat([pd.DataFrame(x) for x in InputDF[column]], keys=InputDF.index).reset_index(level=1,drop=True)
    return df

In [ ]:
def WriteReadCSV(filename, URL):
    write_df = pd.DataFrame(OpenAlexAPI(URL, parameters))
    write_df.to_csv(filename)
    
    read_df = pd.read_csv(filename, sep=',',on_bad_lines='skip', index_col=False, dtype='unicode')
    read_df = read_df.drop(columns=['Unnamed: 0'])
    
    return read_df

In [ ]:
df = WriteReadCSV('OpenAlexData.csv', 'https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i57206974|i130238516|i2801744472|i27837315|i157725225|i2802508227|i4210135927|i188538660|i205783295|i859038795|i135310074|i20089843|i130769515|i51556381|i79576946|i170897317|i204465549|i169521973|i114395901')

In [ ]:
MN_df = WriteReadCSV('OpenAlexMNData.csv', 'https://api.openalex.org/works?page=1&filter=type:types/dataset,authorships.institutions.lineage:i130238516')


In [ ]:
pd.set_option("display.max_columns", 55)
read_df.columns
